# Indirect losses: single-basin vs multi-basin flooding


| manuscript | figure file | what |
|---|---|---|
| Main text, `fig:res2` | `fig_singlevsmulti_short.png` | simultaneous multi-basin vs sum of single-basin losses, 4–28 days, log scale |
| Supplementary, `fig:si_singlevsmulti` | `fig_singlevsmulti_merge.png` | the same comparison per duration, with each basin shown separately (outliers removed) |

**Data**: `data/monte_carlo/cambodia_simulations_results_230927_singlevsmulti.csv` (2–12 days)
and `..._singlevsmulti_v2.csv` (16–28 days); 100 Monte Carlo realizations per scenario
and duration.

**Environment**: `dsc` conda env (pandas, numpy, plotly, kaleido).

In [1]:
from pathlib import Path

BUNDLE = Path.cwd()
MC = BUNDLE / "data" / "monte_carlo"
SCEN = BUNDLE / "data" / "scenarios"
REF = BUNDLE / "data" / "reference"
FIGDIR = BUNDLE / "figures" / "generated"
FIGDIR.mkdir(parents=True, exist_ok=True)


def save_plotly(fig, name):
    """Interactive HTML always; PNG at the layout size if the `kaleido` package is installed."""
    fig.write_html(FIGDIR / f"{name}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(FIGDIR / f"{name}.png")
    except Exception:
        pass

## Setup and data

In [2]:
# All imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Color scheme for consistency
colors = {
    'single_basin_mekong': '#1f77b4',
    'single_basin_saigon': '#ff7f0e', 
    'single_basin_tonle_sap': '#2ca02c',
    'cascading_hazards_0d': '#d62728',
    'sum_single_basins': '#9467bd'
}

In [3]:
# Load datasets
df1 = pd.read_csv(MC / 'cambodia_simulations_results_230927_singlevsmulti.csv')
df2 = pd.read_csv(MC / 'cambodia_simulations_results_230927_singlevsmulti_v2.csv')

# Define analysis parameters
target_scenarios = ['single_basin_mekong', 'single_basin_saigon', 'single_basin_tonle_sap', 'cascading_hazards_0d']
short_durations = [2, 4, 8, 12]  # Short duration disruptions
long_durations = [16, 20, 24, 28]  # Long duration disruptions

# Filter datasets
df1_filtered = df1[df1['simulation_name'].isin(target_scenarios) & df1['duration'].isin(short_durations)].copy()
df2_filtered = df2[df2['simulation_name'].isin(target_scenarios) & df2['duration'].isin(long_durations)].copy()

# Print summary statistics
print("="*80)
print("DATA SUMMARY")
print("="*80)
print(f"\nDataset 1 (short durations): {df1_filtered.shape[0]} rows × {df1_filtered.shape[1]} columns")
print(f"  Short durations: {short_durations}")
print(f"  Scenarios: {target_scenarios}")
print(f"  Counts per scenario per duration:")
for dur in short_durations:
    print(f"    {dur} days: {df1_filtered[df1_filtered['duration'] == dur].groupby('simulation_name').size().to_dict()}")

print(f"\nDataset 2 (long durations): {df2_filtered.shape[0]} rows × {df2_filtered.shape[1]} columns")
print(f"  Long durations: {long_durations}")
print(f"  Scenarios: {target_scenarios}")
print(f"  Counts per scenario per duration:")
for dur in long_durations:
    print(f"    {dur} days: {df2_filtered[df2_filtered['duration'] == dur].groupby('simulation_name').size().to_dict()}")

print("\n" + "="*80)

DATA SUMMARY

Dataset 1 (short durations): 1600 rows × 17 columns
  Short durations: [2, 4, 8, 12]
  Scenarios: ['single_basin_mekong', 'single_basin_saigon', 'single_basin_tonle_sap', 'cascading_hazards_0d']
  Counts per scenario per duration:
    2 days: {'cascading_hazards_0d': 100, 'single_basin_mekong': 100, 'single_basin_saigon': 100, 'single_basin_tonle_sap': 100}
    4 days: {'cascading_hazards_0d': 100, 'single_basin_mekong': 100, 'single_basin_saigon': 100, 'single_basin_tonle_sap': 100}
    8 days: {'cascading_hazards_0d': 100, 'single_basin_mekong': 100, 'single_basin_saigon': 100, 'single_basin_tonle_sap': 100}
    12 days: {'cascading_hazards_0d': 100, 'single_basin_mekong': 100, 'single_basin_saigon': 100, 'single_basin_tonle_sap': 100}

Dataset 2 (long durations): 1600 rows × 17 columns
  Long durations: [16, 20, 24, 28]
  Scenarios: ['single_basin_mekong', 'single_basin_saigon', 'single_basin_tonle_sap', 'cascading_hazards_0d']
  Counts per scenario per duration:
    1

In [4]:
def calculate_sum_single_basins(df, duration, single_scenarios=['single_basin_mekong', 'single_basin_saigon', 'single_basin_tonle_sap']):
    """
    Calculate sum of single basins for a given duration.
    Uses pivot table to ensure all three basins are present for each mc_repetition.
    This ensures the count is always 100 (or whatever the base count is).
    """
    single_data = df[
        (df['duration'] == duration) & df['simulation_name'].isin(single_scenarios)
    ][['mc_repetition', 'simulation_name', 'household_loss']]
    
    if single_data.empty:
        return pd.Series(dtype=float)
    
    # Pivot to ensure we have all three basins for each repetition
    pivot = single_data.pivot_table(
        index='mc_repetition',
        columns='simulation_name',
        values='household_loss',
        aggfunc='first'
    )
    
    # Require all three basins to be present
    valid = pivot.dropna(subset=single_scenarios)
    
    if valid.empty:
        return pd.Series(dtype=float)
    
    # Sum across the three basins
    return valid[single_scenarios].sum(axis=1)

In [5]:
def _yrange(arr):
    if len(arr) == 0:
        return [0, 1]
    lo, hi = float(np.min(arr)), float(np.max(arr))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        pad = 1.0 if not np.isfinite(lo) or not np.isfinite(hi) else max(1e-9, 0.05 * hi)
        return [0, hi + pad]
    pad = 0.05 * (hi - lo if hi != lo else hi or 1.0)
    return [lo - pad, hi + pad]

### Single basins vs Multi-basins simulations

No outliers removal

In [6]:
# Prepare data for compact boxplots
short_durations_filtered = [4, 8, 12]
all_durations = short_durations_filtered + long_durations

# Combine datasets
df_combined = pd.concat([df1_filtered, df2_filtered], ignore_index=True)

# Prepare data WITH OUTLIERS
data_with_outliers = []
for duration in all_durations:
    sim_data = df_combined[(df_combined['duration'] == duration) & 
                           (df_combined['simulation_name'] == 'cascading_hazards_0d')]['household_loss'].values
    sum_data = calculate_sum_single_basins(df_combined, duration).values
    data_with_outliers.append({
        'duration': duration, 
        'simultaneous': sim_data, 
        'sum_single': sum_data
    })

# PLOT 1: WITH OUTLIERS (log scale)
fig1 = go.Figure()

dx = 0.8  # horizontal offset so boxes are close but not on top
w = 1.5   # box width (narrow to avoid overlap)

for i, d in enumerate(data_with_outliers):
    dur = d['duration']
    
    # Simultaneous (left)
    fig1.add_trace(go.Box(
        x=[dur - dx] * len(d['simultaneous']),
        y=d['simultaneous'],
        name='Simultaneous',
        marker_color=colors['cascading_hazards_0d'],
        boxpoints='outliers',
        width=w,
        showlegend=(i == 0)
    ))
    
    # Sum of Single (right)
    fig1.add_trace(go.Box(
        x=[dur + dx] * len(d['sum_single']),
        y=d['sum_single'],
        name='Sum Single',
        marker_color=colors['sum_single_basins'],
        boxpoints='outliers',
        width=w,
        showlegend=(i == 0)
    ))

fig1.update_xaxes(
    title_text='Disruption Duration [days]',
    tickmode='array',
    tickvals=all_durations,
    ticktext=[str(d) for d in all_durations],
    title_font=dict(size=28),
    tickfont=dict(size=24)
)

fig1.update_yaxes(
    title_text='Indirect Losses [M$] (log scale)',
    type='log',
    title_font=dict(size=28),
    tickfont=dict(size=24)
)

fig1.update_layout(
    height=800, width=1800,
    template='plotly_white',
    showlegend=True,
    legend=dict(x=0.02, y=0.98, font=dict(size=24)),
)

save_plotly(fig1, 'fig_singlevsmulti_short')
fig1.show()

/var/folders/q5/j53bmmsn5_v21fb0ltm6q7w40000gn/T/ipykernel_44966/766761587.py:15: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [15]:
all_durations_with_2 = [2] + all_durations  # [2, 4, 8, 12, 16, 20, 24, 28]
SCALE = 6  # 6x the x units; tick labels still show the real duration

data_with_2d = []
for duration in all_durations_with_2:
    sim_data = df_combined[(df_combined['duration'] == duration) &
                           (df_combined['simulation_name'] == 'cascading_hazards_0d')]['household_loss'].values
    sum_data = calculate_sum_single_basins(df_combined, duration).values
    data_with_2d.append({
        'duration': duration,
        'simultaneous': sim_data,
        'sum_single': sum_data
    })

fig_2d = go.Figure()

dx = 2.4  # horizontal offset between paired boxes (3x wider)
w = 4.5   # box width (3x wider)

for i, d in enumerate(data_with_2d):
    x_center = d['duration'] * SCALE

    fig_2d.add_trace(go.Box(
        x=[x_center - dx] * len(d['simultaneous']),
        y=d['simultaneous'],
        name='Simultaneous',
        marker_color=colors['cascading_hazards_0d'],
        boxpoints='outliers',
        width=w,
        showlegend=(i == 0)
    ))

    fig_2d.add_trace(go.Box(
        x=[x_center + dx] * len(d['sum_single']),
        y=d['sum_single'],
        name='Sum Single',
        marker_color=colors['sum_single_basins'],
        boxpoints='outliers',
        width=w,
        showlegend=(i == 0)
    ))

fig_2d.update_xaxes(
    title_text='Disruption Duration [days]',
    type='linear',  # force linear so `width` is in data units
    tickmode='array',
    tickvals=[d * SCALE for d in all_durations_with_2],
    ticktext=[str(d) for d in all_durations_with_2],
    title_font=dict(size=28),
    tickfont=dict(size=24)
)

fig_2d.update_yaxes(
    title_text='Indirect Losses [M$] (log scale)',
    type='log',
    title_font=dict(size=28),
    tickfont=dict(size=24)
)

fig_2d.update_layout(
    height=800, width=1800,
    template='plotly_white',
    showlegend=True,
    legend=dict(x=0.02, y=0.98, font=dict(size=24)),
)

save_plotly(fig_2d, 'fig_singlevsmulti_short_with_2d')
fig_2d.show()


/var/folders/q5/j53bmmsn5_v21fb0ltm6q7w40000gn/T/ipykernel_44966/766761587.py:15: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




### Single basins vs Multi-basins (multiple durations)

In [8]:
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=[f'{d}-days disruption' for d in short_durations] + [f'{d}-days disruption' for d in long_durations],
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

single_scenarios = ['single_basin_mekong', 'single_basin_saigon', 'single_basin_tonle_sap']
x_labels = ['Mekong', 'Saigon', 'Tonlè Sap', '', 'Simultaneous', 'Sum Single']

# Panel A positions (rows 1-2): Short durations
panel_a_positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
y_data_upper_a, y_data_lower_a = [], []

for idx, duration in enumerate(short_durations):
    row, col = panel_a_positions[idx]
    duration_data = df1_filtered[df1_filtered['duration'] == duration]
    x_pos = 0

    for scenario in single_scenarios:
        scenario_data = duration_data[duration_data['simulation_name'] == scenario]['household_loss']
        if len(scenario_data) > 0:
            fig.add_trace(
                go.Box(
                    y=scenario_data,
                    marker_color=colors[scenario],
                    showlegend=False,
                    boxpoints='outliers',
                    offsetgroup=x_pos,
                    x0=x_pos
                ),
                row=row, col=col
            )
            (y_data_upper_a if idx < 2 else y_data_lower_a).extend(scenario_data.values)
            x_pos += 1

    x_pos += 1  # Skip for dashed line

    cascading_data = duration_data[duration_data['simulation_name'] == 'cascading_hazards_0d']['household_loss']
    if len(cascading_data) > 0:
        fig.add_trace(
            go.Box(
                y=cascading_data,
                marker_color=colors['cascading_hazards_0d'],
                showlegend=False,
                boxpoints='outliers',
                offsetgroup=x_pos,
                x0=x_pos
            ),
            row=row, col=col
        )
        (y_data_upper_a if idx < 2 else y_data_lower_a).extend(cascading_data.values)
        x_pos += 1

    sum_data = calculate_sum_single_basins(df1_filtered, duration, single_scenarios)
    if not sum_data.empty:
        fig.add_trace(
            go.Box(
                y=sum_data.values,
                marker_color=colors['sum_single_basins'],
                showlegend=False,
                boxpoints='outliers',
                offsetgroup=x_pos,
                x0=x_pos
            ),
            row=row, col=col
        )
        (y_data_upper_a if idx < 2 else y_data_lower_a).extend(sum_data.values)

# Panel B positions (rows 3-4): Long durations
panel_b_positions = [(3, 1), (3, 2), (4, 1), (4, 2)]
y_data_upper_b, y_data_lower_b = [], []

for idx, duration in enumerate(long_durations):
    row, col = panel_b_positions[idx]
    duration_data = df2_filtered[df2_filtered['duration'] == duration]
    x_pos = 0

    for scenario in single_scenarios:
        scenario_data = duration_data[duration_data['simulation_name'] == scenario]['household_loss']
        if len(scenario_data) > 0:
            fig.add_trace(
                go.Box(
                    y=scenario_data,
                    marker_color=colors[scenario],
                    showlegend=False,
                    boxpoints='outliers',
                    offsetgroup=x_pos,
                    x0=x_pos
                ),
                row=row, col=col
            )
            (y_data_upper_b if idx < 2 else y_data_lower_b).extend(scenario_data.values)
            x_pos += 1

    x_pos += 1  # Skip for dashed line

    cascading_data = duration_data[duration_data['simulation_name'] == 'cascading_hazards_0d']['household_loss']
    if len(cascading_data) > 0:
        fig.add_trace(
            go.Box(
                y=cascading_data,
                marker_color=colors['cascading_hazards_0d'],
                showlegend=False,
                boxpoints='outliers',
                offsetgroup=x_pos,
                x0=x_pos
            ),
            row=row, col=col
        )
        (y_data_upper_b if idx < 2 else y_data_lower_b).extend(cascading_data.values)
        x_pos += 1

    sum_data = calculate_sum_single_basins(df2_filtered, duration, single_scenarios)
    if not sum_data.empty:
        fig.add_trace(
            go.Box(
                y=sum_data.values,
                marker_color=colors['sum_single_basins'],
                showlegend=False,
                boxpoints='outliers',
                offsetgroup=x_pos,
                x0=x_pos
            ),
            row=row, col=col
        )
        (y_data_upper_b if idx < 2 else y_data_lower_b).extend(sum_data.values)

# Calculate y-ranges
upper_range_a = _yrange(np.array(y_data_upper_a))
lower_range_a = _yrange(np.array(y_data_lower_a))
upper_range_b = _yrange(np.array(y_data_upper_b))
lower_range_b = _yrange(np.array(y_data_lower_b))

# Panel A: Update axes and add backgrounds + dashed separator
for idx in range(4):
    row, col = panel_a_positions[idx]
    y_range = upper_range_a if idx < 2 else lower_range_a

    fig.add_shape(
        type="rect",
        x0=-0.5, y0=y_range[0],
        x1=5.5, y1=y_range[1],
        fillcolor="lightgrey",
        opacity=0.15,
        layer="below",
        line_width=0,
        row=row, col=col
    )

    fig.add_shape(
        type="line",
        x0=3, y0=y_range[0],
        x1=3, y1=y_range[1],
        line=dict(color="gray", width=2, dash="dash"),
        row=row, col=col
    )

    fig.update_xaxes(tickvals=[0, 1, 2, 3, 4, 5], ticktext=x_labels, row=row, col=col)
    fig.update_yaxes(range=y_range, row=row, col=col)
    if col == 1:
        fig.update_yaxes(title_text='Indirect Losses [M$]', row=row, col=col)

# Panel B: Update axes and add backgrounds + dashed separator
for idx in range(4):
    row, col = panel_b_positions[idx]
    y_range = upper_range_b if idx < 2 else lower_range_b

    fig.add_shape(
        type="rect",
        x0=-0.5, y0=y_range[0],
        x1=5.5, y1=y_range[1],
        fillcolor="lightgrey",
        opacity=0.15,
        layer="below",
        line_width=0,
        row=row, col=col
    )

    fig.add_shape(
        type="line",
        x0=3, y0=y_range[0],
        x1=3, y1=y_range[1],
        line=dict(color="gray", width=2, dash="dash"),
        row=row, col=col
    )

    fig.update_xaxes(tickvals=[0, 1, 2, 3, 4, 5], ticktext=x_labels, row=row, col=col)
    fig.update_yaxes(range=y_range, row=row, col=col)
    if col == 1:
        fig.update_yaxes(title_text='Indirect Losses [M$]', row=row, col=col)

# Add A) and B) panel labels
fig.add_annotation(
    text="A)",
    xref="paper", yref="paper",
    x=-0.05, y=1.05,
    showarrow=False,
    font=dict(size=24, family="Arial Black"),
    xanchor='left'
)

fig.add_annotation(
    text="B)",
    xref="paper", yref="paper",
    x=-0.05, y=0.5,
    showarrow=False,
    font=dict(size=24, family="Arial Black"),
    xanchor='left'
)

fig.update_layout(
    height=1800,
    width=1400,
    showlegend=False,
    template='plotly_white'
)

save_plotly(fig, 'fig_singlevsmulti_merge_with_outliers')
fig.show()


/var/folders/q5/j53bmmsn5_v21fb0ltm6q7w40000gn/T/ipykernel_44966/766761587.py:15: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).





#### Modified Z-Score Method

We use the **Modified Z-score** approach instead of the standard Z-score because it's more robust to outliers:

Standard Z-score (sensitive to outliers): Z = (x - mean) / std
Both mean and standard deviation are heavily influenced by outliers themselves.


Modified Z = 0.6745 × (x - median) / MAD
Where MAD (Median Absolute Deviation) = median(|x - median|)

Detection threshold: Modified Z > 3.5
- More conservative than the standard Z > 3 threshold
- Flags only the most extreme values (roughly equivalent to >3.5 standard deviations in a normal distribution)
- Balances removing genuine artifacts while preserving legitimate extreme outcomes

Outliers are detected and removed separately for each combination of:
- Duration (2, 4, 8, 12, 16, 20, 24, 28 days)
- Scenario (Mekong, Saigon, Tonlè Sap, Simultaneous)


In questo scenario, cioè rimuovendo gli outliers 'sum of single' is higher than 'simultaneous' for longer disruptions.

In [9]:
def flag_outliers_modified_z(group: pd.DataFrame, col: str = "household_loss", threshold: float = 3.5):
    """
    Flag outliers using Modified Z-score method.
    More robust to outliers than standard Z-score as it uses median and MAD.
    """
    x = pd.to_numeric(group[col], errors="coerce")
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad == 0:
        mad = 1e-12  # avoid division by zero
    mz = 0.6745 * (x - med) / mad
    group["_outlier"] = np.abs(mz) > threshold
    return group

# Apply outlier detection to both datasets
df1_flagged = (
    df1_filtered
    .groupby(["duration", "simulation_name"], group_keys=False)
    .apply(flag_outliers_modified_z, col="household_loss", threshold=3.5)
)

df2_flagged = (
    df2_filtered
    .groupby(["duration", "simulation_name"], group_keys=False)
    .apply(flag_outliers_modified_z, col="household_loss", threshold=3.5)
)

# Report outliers removed
print("="*80)
print("OUTLIERS REMOVED - SHORT DURATIONS")
print("="*80)
dropped_stats1 = (
    df1_flagged.groupby(["duration", "simulation_name"])["_outlier"]
    .agg(count="size", dropped="sum")
    .assign(share_dropped=lambda d: d["dropped"] / d["count"])
)
print(dropped_stats1)

print("\n" + "="*80)
print("OUTLIERS REMOVED - LONG DURATIONS")
print("="*80)
dropped_stats2 = (
    df2_flagged.groupby(["duration", "simulation_name"])["_outlier"]
    .agg(count="size", dropped="sum")
    .assign(share_dropped=lambda d: d["dropped"] / d["count"])
)
print(dropped_stats2)

# Create cleaned datasets
df1_clean = df1_flagged.loc[~df1_flagged["_outlier"]].drop(columns="_outlier").reset_index(drop=True)
df2_clean = df2_flagged.loc[~df2_flagged["_outlier"]].drop(columns="_outlier").reset_index(drop=True)

print(f"\n\nDataset 1 cleaned: {df1_filtered.shape} -> {df1_clean.shape}")
print(f"Dataset 2 cleaned: {df2_filtered.shape} -> {df2_clean.shape}")

OUTLIERS REMOVED - SHORT DURATIONS
                                 count  dropped  share_dropped
duration simulation_name                                      
2        cascading_hazards_0d      100        1           0.01
         single_basin_mekong       100        0           0.00
         single_basin_saigon       100        0           0.00
         single_basin_tonle_sap    100        0           0.00
4        cascading_hazards_0d      100        0           0.00
         single_basin_mekong       100        2           0.02
         single_basin_saigon       100        1           0.01
         single_basin_tonle_sap    100        2           0.02
8        cascading_hazards_0d      100        1           0.01
         single_basin_mekong       100        8           0.08
         single_basin_saigon       100        6           0.06
         single_basin_tonle_sap    100        3           0.03
12       cascading_hazards_0d      100        2           0.02
         single_basi

/var/folders/q5/j53bmmsn5_v21fb0ltm6q7w40000gn/T/ipykernel_44966/3543793065.py:17: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/q5/j53bmmsn5_v21fb0ltm6q7w40000gn/T/ipykernel_44966/3543793065.py:23: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [10]:
df_combined_clean = pd.concat([df1_clean, df2_clean], ignore_index=True)

data_no_outliers = []
for duration in all_durations:
    sim_data = df_combined_clean[(df_combined_clean['duration'] == duration) &
                                 (df_combined_clean['simulation_name'] == 'cascading_hazards_0d')]['household_loss'].values
    sum_data = calculate_sum_single_basins(df_combined_clean, duration).values
    data_no_outliers.append({
        'duration': duration,
        'simultaneous': sim_data,
        'sum_single': sum_data
    })

fig1_clean = go.Figure()

dx = 0.8
w = 1.5

for i, d in enumerate(data_no_outliers):
    dur = d['duration']

    fig1_clean.add_trace(go.Box(
        x=[dur - dx] * len(d['simultaneous']),
        y=d['simultaneous'],
        name='Simultaneous',
        marker_color=colors['cascading_hazards_0d'],
        boxpoints='outliers',
        width=w,
        showlegend=(i == 0)
    ))

    fig1_clean.add_trace(go.Box(
        x=[dur + dx] * len(d['sum_single']),
        y=d['sum_single'],
        name='Sum Single',
        marker_color=colors['sum_single_basins'],
        boxpoints='outliers',
        width=w,
        showlegend=(i == 0)
    ))

fig1_clean.update_xaxes(
    title_text='Disruption Duration [days]',
    tickmode='array',
    tickvals=all_durations,
    ticktext=[str(d) for d in all_durations],
    title_font=dict(size=28),
    tickfont=dict(size=24)
)

fig1_clean.update_yaxes(
    title_text='Indirect Losses [M$] (log scale)',
    type='log',
    title_font=dict(size=28),
    tickfont=dict(size=24)
)

fig1_clean.update_layout(
    height=800, width=1800,
    template='plotly_white',
    showlegend=True,
    legend=dict(x=0.02, y=0.98, font=dict(size=24)),
)

save_plotly(fig1_clean, 'fig_singlevsmulti_short_no_outliers')
fig1_clean.show()


/var/folders/q5/j53bmmsn5_v21fb0ltm6q7w40000gn/T/ipykernel_44966/766761587.py:15: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [11]:
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=[f'{d}-days disruption' for d in short_durations] + [f'{d}-days disruption' for d in long_durations],
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

single_scenarios = ['single_basin_mekong', 'single_basin_saigon', 'single_basin_tonle_sap']
x_labels = ['Mekong', 'Saigon', 'Tonlè Sap', '', 'Simultaneous', 'Sum Single']

# Panel A positions (rows 1-2): Short durations
panel_a_positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
y_data_upper_a, y_data_lower_a = [], []

for idx, duration in enumerate(short_durations):
    row, col = panel_a_positions[idx]
    duration_data = df1_clean[df1_clean['duration'] == duration]
    x_pos = 0
    
    for scenario in single_scenarios:
        scenario_data = duration_data[duration_data['simulation_name'] == scenario]['household_loss']
        if len(scenario_data) > 0:
            fig.add_trace(
                go.Box(
                    y=scenario_data,
                    marker_color=colors[scenario],
                    showlegend=False,
                    boxpoints='outliers',
                    offsetgroup=x_pos,
                    x0=x_pos
                ),
                row=row, col=col
            )
            (y_data_upper_a if idx < 2 else y_data_lower_a).extend(scenario_data.values)
            x_pos += 1
    
    x_pos += 1  # Skip for dashed line
    
    cascading_data = duration_data[duration_data['simulation_name'] == 'cascading_hazards_0d']['household_loss']
    if len(cascading_data) > 0:
        fig.add_trace(
            go.Box(
                y=cascading_data,
                marker_color=colors['cascading_hazards_0d'],
                showlegend=False,
                boxpoints='outliers',
                offsetgroup=x_pos,
                x0=x_pos
            ),
            row=row, col=col
        )
        (y_data_upper_a if idx < 2 else y_data_lower_a).extend(cascading_data.values)
        x_pos += 1
    
    sum_data = calculate_sum_single_basins(df1_clean, duration, single_scenarios)
    if not sum_data.empty:
        fig.add_trace(
            go.Box(
                y=sum_data.values,
                marker_color=colors['sum_single_basins'],
                showlegend=False,
                boxpoints='outliers',
                offsetgroup=x_pos,
                x0=x_pos
            ),
            row=row, col=col
        )
        (y_data_upper_a if idx < 2 else y_data_lower_a).extend(sum_data.values)

# Panel B positions (rows 3-4): Long durations
panel_b_positions = [(3, 1), (3, 2), (4, 1), (4, 2)]
y_data_upper_b, y_data_lower_b = [], []

for idx, duration in enumerate(long_durations):
    row, col = panel_b_positions[idx]
    duration_data = df2_clean[df2_clean['duration'] == duration]
    x_pos = 0
    
    for scenario in single_scenarios:
        scenario_data = duration_data[duration_data['simulation_name'] == scenario]['household_loss']
        if len(scenario_data) > 0:
            fig.add_trace(
                go.Box(
                    y=scenario_data,
                    marker_color=colors[scenario],
                    showlegend=False,
                    boxpoints='outliers',
                    offsetgroup=x_pos,
                    x0=x_pos
                ),
                row=row, col=col
            )
            (y_data_upper_b if idx < 2 else y_data_lower_b).extend(scenario_data.values)
            x_pos += 1
    
    x_pos += 1  # Skip for dashed line
    
    cascading_data = duration_data[duration_data['simulation_name'] == 'cascading_hazards_0d']['household_loss']
    if len(cascading_data) > 0:
        fig.add_trace(
            go.Box(
                y=cascading_data,
                marker_color=colors['cascading_hazards_0d'],
                showlegend=False,
                boxpoints='outliers',
                offsetgroup=x_pos,
                x0=x_pos
            ),
            row=row, col=col
        )
        (y_data_upper_b if idx < 2 else y_data_lower_b).extend(cascading_data.values)
        x_pos += 1
    
    sum_data = calculate_sum_single_basins(df2_clean, duration, single_scenarios)
    if not sum_data.empty:
        fig.add_trace(
            go.Box(
                y=sum_data.values,
                marker_color=colors['sum_single_basins'],
                showlegend=False,
                boxpoints='outliers',
                offsetgroup=x_pos,
                x0=x_pos
            ),
            row=row, col=col
        )
        (y_data_upper_b if idx < 2 else y_data_lower_b).extend(sum_data.values)

# Calculate y-ranges
upper_range_a = _yrange(np.array(y_data_upper_a))
lower_range_a = _yrange(np.array(y_data_lower_a))
upper_range_b = _yrange(np.array(y_data_upper_b))
lower_range_b = _yrange(np.array(y_data_lower_b))

# Panel A: Update axes and add backgrounds + dashed separator
for idx in range(4):
    row, col = panel_a_positions[idx]
    y_range = upper_range_a if idx < 2 else lower_range_a
    
    fig.add_shape(
        type="rect",
        x0=-0.5, y0=y_range[0],
        x1=5.5, y1=y_range[1],
        fillcolor="lightgrey",
        opacity=0.15,
        layer="below",
        line_width=0,
        row=row, col=col
    )
    
    fig.add_shape(
        type="line",
        x0=3, y0=y_range[0],
        x1=3, y1=y_range[1],
        line=dict(color="gray", width=2, dash="dash"),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1, 2, 3, 4, 5], ticktext=x_labels, row=row, col=col)
    fig.update_yaxes(range=y_range, row=row, col=col)
    if col == 1:
        fig.update_yaxes(title_text='Indirect Losses [M$]', row=row, col=col)

# Panel B: Update axes and add backgrounds + dashed separator
for idx in range(4):
    row, col = panel_b_positions[idx]
    y_range = upper_range_b if idx < 2 else lower_range_b
    
    fig.add_shape(
        type="rect",
        x0=-0.5, y0=y_range[0],
        x1=5.5, y1=y_range[1],
        fillcolor="lightgrey",
        opacity=0.15,
        layer="below",
        line_width=0,
        row=row, col=col
    )
    
    fig.add_shape(
        type="line",
        x0=3, y0=y_range[0],
        x1=3, y1=y_range[1],
        line=dict(color="gray", width=2, dash="dash"),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1, 2, 3, 4, 5], ticktext=x_labels, row=row, col=col)
    fig.update_yaxes(range=y_range, row=row, col=col)
    if col == 1:
        fig.update_yaxes(title_text='Indirect Losses [M$]', row=row, col=col)

# Add A) and B) panel labels
fig.add_annotation(
    text="A)",
    xref="paper", yref="paper",
    x=-0.05, y=1.05,
    showarrow=False,
    font=dict(size=24, family="Arial Black"),
    xanchor='left'
)

fig.add_annotation(
    text="B)",
    xref="paper", yref="paper",
    x=-0.05, y=0.5,
    showarrow=False,
    font=dict(size=24, family="Arial Black"),
    xanchor='left'
)

fig.update_layout(
    height=1800,
    width=1400,
    showlegend=False,
    template='plotly_white'
)

save_plotly(fig, 'fig_singlevsmulti_merge')
fig.show()

/var/folders/q5/j53bmmsn5_v21fb0ltm6q7w40000gn/T/ipykernel_44966/766761587.py:15: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


